# Optimization methods for MLE

## Data

In [ ]:
import numpy as np
from scipy.optimize import minimize
import warnings

d = np.array([1645, 183, 37, 13, 1, 1])
x = np.repeat(np.arange(1, 7), d)

## Log-likelihood functions

In [ ]:
def neg_ll(par, x):
    """Negative log-likelihood (for minimization)"""
    lam = par[0]
    return -(np.sum(x) * np.log(lam) - len(x) * np.log(np.exp(lam) - 1))

def neg_grad(par, x):
    """Negative gradient"""
    lam = par[0]
    return np.array([-(np.sum(x) / lam - len(x) * np.exp(lam) / (np.exp(lam) - 1))])

## Why do warnings appear?

In [ ]:
## Evaluating at invalid parameter values
with warnings.catch_warnings():
    warnings.simplefilter("always")
    print(f"log(-0.3) = {np.log(-0.3)}")
    print(f"log(exp(-0.3) - 1) = log({np.exp(-0.3) - 1:.4f}) = {np.log(np.exp(-0.3) - 1)}")
    print()
    print(f"log(0.35) = {np.log(0.35):.4f}  -- valid")
    print(f"log(exp(0.35) - 1) = {np.log(np.exp(0.35) - 1):.4f}  -- valid")

## Newton-Raphson

In [ ]:
## Manual Newton-Raphson step from lambda=1
lam0 = 1.0
g = -(neg_grad([lam0], x)[0])  # positive gradient (for maximization)
## Hessian
h = -np.sum(x) / lam0**2 + len(x) * np.exp(lam0) / (np.exp(lam0) - 1)**2

print(f"At lambda = 1.0:")
print(f"  Gradient = {g:.2f} (strongly negative: optimum is to the LEFT)")
print(f"  Hessian  = {h:.2f}")
print(f"  Full Newton step: lambda_new = {lam0} - ({g:.2f} / {h:.2f}) = {lam0 - g/h:.4f}")

## Line search overshoot example
print("If line search tries step size 1.5:")
overshoot = lam0 - 1.5 * g/h
print(f"  lambda_candidate = {overshoot:.4f}")
with warnings.catch_warnings():
    warnings.simplefilter("always")
    val = np.sum(x) * np.log(overshoot) - len(x) * np.log(np.exp(overshoot) - 1)
    print(f"  ll({overshoot:.4f}) = {val}")
    print("  -> If candidate < 0, log() produces NaN, line search backtracks")

## Why BFGS produces more warnings

In [ ]:
## Simulate what BFGS does near lambda = 0.01
eps = 1e-4
lam_small = 0.01
print(f"At lambda = {lam_small}:")
print(f"  ll(lam + eps) = {-neg_ll([lam_small + eps], x):.2f}")
print(f"  ll(lam - eps) = {-neg_ll([lam_small - eps], x):.2f} -- still valid")
print()

lam_tiny = 0.00005
print(f"At lambda = {lam_tiny}:")
with warnings.catch_warnings():
    warnings.simplefilter("always")
    print(f"  ll(lam + eps) = {-neg_ll([lam_tiny + eps], x):.2f}")
    print(f"  ll(lam - eps) = {-neg_ll([lam_tiny - eps], x)} -- NaN!")

## Comparison: Newton-Raphson vs BFGS

In [ ]:
res_nm = minimize(neg_ll, x0=[0.5], args=(x,), method="Nelder-Mead")
res_nm

res_bfgs = minimize(neg_ll, x0=[0.5], args=(x,), method="BFGS", jac=lambda p, x: neg_grad(p, x), options={"disp": False})

res_lbfgsb = minimize(neg_ll, x0=[0.5], args=(x,), method="L-BFGS-B",
                       jac=lambda p, x: neg_grad(p, x), bounds=[(1e-8, None)])
res_lbfgsb

## Fix 1: safeguard in the log-likelihood

In [ ]:
def neg_ll_safe(par, x):
    lam = par[0]
    if lam <= 0:
        return np.inf
    return -(np.sum(x) * np.log(lam) - len(x) * np.log(np.exp(lam) - 1))

res_safe = minimize(neg_ll_safe, x0=[0.5], args=(x,), method="Nelder-Mead")
res_safe

## Fix 2: supply analytical gradient and Hessian

In [ ]:
def neg_hess(par, x):
    lam = par[0]
    return np.array([[np.sum(x) / lam**2 - len(x) * np.exp(lam) / (np.exp(lam) - 1)**2]])

## Safeguarded + exact derivatives
res_exact = minimize(neg_ll_safe, x0=[0.5], args=(x,), method="Newton-CG",
                     jac=lambda p, x: neg_grad(p, x),
                     hessp=lambda p, v, x: neg_hess(p, x) @ v)
res_exact

## Fix 3: use bounded optimization

In [ ]:
res_bounded = minimize(neg_ll, x0=[0.5], args=(x,), method="L-BFGS-B",
                       bounds=[(1e-8, None)])

res_bounded